---
title: "Exercise 9: Differential Analysis"
format: html
editor: source
editor_options: 
  chunk_output_type: console
jupyter: ir
---


## Differential analysis with multiple samples

In this exercise, we extend our analysis to multi-sample, multi-condition spatial transcriptomics data. With multiple samples per condition, for example, 2 colorectal carcinoma (CRC) samples and 2 normal adjacent tissue (NAT) samples, we can ask not only: 

> "Is a gene spatially variable within a tissue?" 

but also: 

> "Does its spatial pattern change between conditions?" 

Genes showing such changes are called **differential spatial patterns (DSP)** genes. Using the `DESpace` package, we will learn how to identify such genes, which may be spatially variable in one condition but not in another, or spatially variable in both conditions but with different spatial patterns.


## Learning Objectives

By the end of this exercise, you will be able to:

  - Understand the concept of DSP
  - Perform global DSP tests across multiple samples  
  - Perform individual-cluster tests to find cluster-specific DSP genes  
  - Interpret the DSP patterns 

# Libraries

In [ ]:
#| warning: false
#| output: false

library(alabaster.sfe)
library(SpatialExperiment)
library(ggspavis)
library(patchwork)
library(DESpace)

## Load preprocessed data

We will work with the same VisiumHD dataset from the human colon cancer study [recently published](https://www.nature.com/articles/s41588-025-02193-3). Here we use a subsetted dataset containing 4 slides from three patients (P1CRC, P5CRC, P3NAT, and P5NAT), representing either colorectal carcinoma (CRC) or normal adjacent tissue (NAT), all with pre-computed clusters.

In [ ]:
spe <- readObject("data/day3/03.1_sfeMultNorm")

# Extract sample labels
sample_id <- colData(spe)[["sample_id"]]

# Create a sample indicator matrix
# Each column represents a sample; entries are 1 if the spot belongs to that sample
sample_mat <- model.matrix(~ sample_id - 1)
 
# Total counts of each gene in each sample
gene_counts <- counts(spe)
counts_per_sample <- gene_counts %*% sample_mat

# Number of non-zero spots per gene per sample
nonzero_per_sample <- (gene_counts > 0) %*% sample_mat 

# Keep only genes that pass the filter in all samples
sel_matrix <- t(counts_per_sample >= 50 & nonzero_per_sample >= 20)
sel <- colMeans(sel_matrix) == 1
spe <- spe[sel, ]

`colData(spe)` includes: 

  - `sample_id`: sample ID 
  - `condition`: CRC vs. Normal tissue 
  - `cluster`: spatial domain labels

In [ ]:
head(colData(spe), 3)
table(spe$condition)

Visualize clusters for each sample.

In [ ]:
samples <- sort(unique(spe$sample_id))
lapply(samples, \(.)
       plotCoords(spe[, spe$sample_id == .],
                  annotate="cluster",
                  in_tissue=NULL,
                  point_size = 0.1) +
         ggtitle(.)) |>
  wrap_plots(nrow = 2) &
  scale_color_manual(values=unname(pals::trubetskoy())) &
  theme_void() &
  theme(legend.position = "none")

## DESpace

### Global test

To answer the question: does the expression of a gene change between conditions, but change differently for some domains compared to others?

`DESpace::dsp_test()` first aggregates spot- or cell-level counts into pseudobulk counts for each (sample, domain) combination, then tests whether the interaction term (condition x domain) in the model is different from zero.

> This may take a few minutes to run. While it is running, take a look at the [package vignette](https://peicai.github.io/DESpace/articles/DSP.html).

In [ ]:
res_edgeR <- dsp_test(spe,
                      cluster_col = "cluster",
                      sample_col = "sample_id",
                      condition_col = "condition",
                      verbose = TRUE)

::: callout-important
## Exercise 1

Check out the results:

- Extract gene-level results.
- Count how many DSP genes are significant at 5\% FDR.
- Take the top 3 DSP genes and visualize their spatial expression across all samples using `ggspavis::plotCoords()`.
- How would you describe the spatial expression patterns of these genes? To help with interpretation, try looking up information about these genes (e.g., UniProt or the Human Protein Atlas) to get clues about which cell types or tissue domains they may highlight.
:::

::: {.callout-tip collapse="true"}
## Answer

In [ ]:
# Extract gene-level results
dsp_global <- res_edgeR$gene_results
head(dsp_global, 3)

# Count significant DSP genes (at 5% FDR significance level)
table(dsp_global$FDR <= 0.05)

# Top 3 DSP genes
gs <- rownames(dsp_global)[seq_len(3)]
gs

In [ ]:
plots <- lapply(samples, \(s) {
  # Subset to sample s
  spe_j <- spe[, spe$sample_id == s]
  
  # Plot cluster 
  p_cluster <- plotCoords(
    spe_j,
    annotate = "cluster",
    in_tissue = NULL,
    point_size = 0.05,
    legend_position = "none") +
    ggtitle(s) +
    scale_color_manual(values = unname(pals::trubetskoy()))
  
  # Plots for each DSP gene
  p_genes <- lapply(gs, \(g) {
    plotCoords(
      spe_j,
      annotate = g,
      point_size = 0.05,
      in_tissue = NULL,
      assay_name = "logcounts",
      feature_names = "Symbol"
    ) +
      scale_color_gradientn(colors = pals::brewer.reds(n=12))
  })
  
  # combine the cluster plot and 3 gene plots
  c(p_cluster, p_genes)
})

plots_flat <- unlist(plots, recursive = FALSE)
wrap_plots(plots_flat, ncol = 4)

:::

### Individual domain test

The global test tells us which genes show different spatial patterns globally, but it does not tell us which clusters drive the differences.

To identify the key spatial domain where expression changes across conditions, we use `DESpace::individual_dsp()`, which performs domain-specific DSP tests.

In [ ]:
# Focus on one cluster to decrease runtime in this exercise
spe$c8 <- factor(ifelse(spe$cluster == "8", 1, 0))
dsp_clu <- individual_dsp(spe,
                      cluster_col = "c8",
                      sample_col = "sample_id",
                      condition_col = "condition",
                      filter_gene = FALSE,
                      test = "LRT")

::: callout-important
## Exercise 2

- Extract the 4th-ranked DSP gene.
- Visualize its expression across samples via `ggspavis::plotCoords()` or `DESpace::FeaturePlot()` and interpret the result.
:::

::: {.callout-tip collapse="true"}
## Answer

In [ ]:
# Select cluster id (because after binarization cluster 8 = "1")
id_clu <- "1"

# DSP results for cluster 8
head(dsp_clu[[id_clu]], 5)
# dsp_clu[["1"]] stores DSP genes for cluster 8  (after binarization)
# dsp_clu[["0"]] stores DSP genes for all other regions (not useful here)
# If full clusters are used originally, dsp_clu[["8"]] would store cluster 8 results

# Select the 4th-ranked DSP gene
(gs <- rownames(dsp_clu[[id_clu]])[4])

In [ ]:
# Rotate slices
colData(spe) <- colData(spe) |> cbind(spatialCoords(spe))
spe$row <- spe$pxl_col_in_fullres
spe$col <- -spe$pxl_row_in_fullres

# Visualization
plots <- lapply(samples, \(.) {
  # Subset sample
  spe_j <- spe[, colData(spe)$sample_id == .]
  # FeaturePlot for cluster 8
  plot <- FeaturePlot(spe_j, 
                      feature = gs, 
                      coordinates = c("pxl_row_in_fullres", "pxl_col_in_fullres"),
                      cluster_col = "c8", # binary cluster column
                      cluster = "1",      # plot on cluster 8
                      platform = "VisiumHD",
                      sf_dim = 400,
                      diverging = TRUE,
                      point_size = 0.05,
                      linewidth = 0.6) +
    theme(legend.position = "right") +
    labs(color = "") + ggtitle(.) 
  
  return(plot)
})
wrap_plots(plots, ncol = 2) + plot_layout(guides = 'collect') + ggtitle(gs)

In CRC samples, COL12A1 shows higher expression in cluster 8 and covers a larger region.
In normal samples, its expression is weaker and more limited.

Cluster 8 might correspond to a stromal region that becomes expanded or remodeled in CRC.
`individual_dsp()` helps identify which spatial domain shows condition-specific changes in gene expression.
:::

::: callout-important
## Bonus question
Try running the individual domain test using the default setting `test = "QLF"`. How do the results change? (Hint: check the number of significant genes.) Why might this happen?
:::

Clear your environment:

In [ ]:
#| eval: false
rm(list = ls())
gc()
.rs.restartR()

:::{.callout-important}
**Key Takeaways:**

- DSP genes capture changes in where a gene is expressed across conditions, not only how much.

- `dsp_test()` identifies global DSP genes across all clusters and samples.

- `individual_dsp()` pinpoints which clusters (spatial domains) drive these differences.
:::